# BacDive — Bacterial Diversity Metadatabase: Data Ingestion

**BacDive** (Bacterial Diversity Metadatabase) is the world's largest database for standardised bacterial and archaeal phenotypic information, maintained by the **DSMZ** (German Collection of Microorganisms and Cell Cultures). It integrates strain-level data from culture collections with published literature, providing a central resource for microbial diversity.

Key data types:
| Data type | Description |
|---|---|
| **Taxonomy** | NCBI-aligned classification from domain to species/strain |
| **Morphology** | Cell shape, flagella, spore formation, Gram stain |
| **Physiology** | Oxygen tolerance, temperature/pH range and optimum, NaCl tolerance |
| **Isolation source** | Habitat, country, sample material |
| **Culture & growth** | Media, incubation conditions, colony characteristics |
| **Genome** | Assembly accessions, GC content, genome size |
| **Sequence** | 16S rRNA accessions linked to SILVA/ENA |

**API base:** `https://bacdive.dsmz.de/api/bacdive/`  
**Authentication:** HTTP Basic Auth (public demo: `api@bacdive.dsmz.de` / `api`)  
**Docs:** https://api.bacdive.dsmz.de/

**Reference:** Reimer et al. (2022), *Nucleic Acids Research*, BacDive in 2022: the knowledge base for standardized bacterial and archaeal phenotypic information.

# TODO

* [x] **Ingest data**
    * [x] Connect to BacDive API with basic auth and confirm access
    * [x] Fetch a batch of strains via the paginated endpoint (≥10 pages, cached)
    * [x] Flatten nested JSON into a Polars DataFrame with key phenotypic fields
    * [x] Fetch detailed records for a genus of interest (*Escherichia*) via taxon endpoint
    * [x] Print shape, dtypes, and head of the resulting DataFrames
* [ ] **Explore and clean**
    * [ ] Summarize missingness across phenotypic columns
    * [ ] Examine distributions of Gram stain, oxygen tolerance, and isolation source
    * [ ] Parse and validate numeric columns (temperature optimum, pH optimum)
* [ ] **Analysis**
    * [ ] Compare physiology profiles across major taxonomic groups (phylum/class)
    * [ ] Identify co-occurrence patterns between oxygen tolerance and temperature range
    * [ ] Cluster strains by phenotypic profile (PCA / UMAP on encoded trait matrix)
* [ ] **Visualization**
    * [ ] Heatmap of phenotypic trait completeness by taxonomic group
    * [ ] Boxplots of temperature/pH optimum by oxygen tolerance category
    * [ ] Geographic map of isolation sources (country-level)
* [ ] **Statistical analysis**
    * [ ] Test whether Gram-positive and Gram-negative strains differ in pH/temperature optima
    * [ ] Discuss multiple hypothesis correction when comparing many trait pairs
    * [ ] Primer on the statistics of trait–trait association in microbial diversity databases

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to BacDive API and Confirm Access

In [ ]:
BACDIVE_BASE = "https://bacdive.dsmz.de/api/bacdive"
# Public demo credentials — documented at https://api.bacdive.dsmz.de/
BACDIVE_AUTH = ("api@bacdive.dsmz.de", "api")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def bacdive_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the BacDive REST API using basic auth.

    Parameters
    ----------
    endpoint : str
        API path relative to BACDIVE_BASE (e.g. "bacdive/533/").
    params : dict, optional
        Query parameters appended to the URL.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{BACDIVE_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, auth=BACDIVE_AUTH, timeout=30)
    resp.raise_for_status()
    return resp.json()


# Connectivity check: fetch BacDive strain 533 (Escherichia coli K-12)
sample = bacdive_get("bacdive/533/")

# The API returns a paginated envelope even for single-ID lookups
first = sample["results"][0]

# Extract species name from the Name and taxonomic classification section
name_block = first.get("Name and taxonomic classification", {})
species = name_block.get("species", {})
full_name = f"{species.get('genus', '')} {species.get('species', '')}".strip()
strain_id = first.get("General", {}).get("BacDive-ID")

print(f"BacDive ID   : {strain_id}")
print(f"Species      : {full_name}")
print(f"Strain desig.: {name_block.get('strain designation', 'n/a')}")
print(f"Top-level keys in record: {list(first.keys())}")

### 1.2 Fetch a Batch of Strains via the Paginated Endpoint

In [ ]:
STRAINS_CACHE = DATA_DIR / "bacdive_strains.json"
MAX_PAGES = 10   # each page contains up to 100 strain records


def fetch_strain_pages(
    cache_path: Path = STRAINS_CACHE,
    max_pages: int = MAX_PAGES,
) -> list[dict]:
    """
    Fetch strain records from the BacDive paginated ``/bacdive/`` endpoint
    and cache the raw JSON to disk.  Subsequent calls load from cache.

    Parameters
    ----------
    cache_path : Path
        File path for the JSON cache.
    max_pages : int
        Maximum number of pages to retrieve (each page ≤ 100 records).

    Returns
    -------
    list[dict]
        Raw strain record dicts as returned by the BacDive API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_strains: list[dict] = []
    page = 1   # BacDive pagination is 1-indexed

    while page <= max_pages:
        resp = bacdive_get("bacdive/", params={"page": page})
        batch = resp.get("results", [])
        if not batch:
            break   # no more records

        all_strains.extend(batch)
        total = resp.get("count", "?")
        print(f"  Page {page:>3} — {len(all_strains):>5} / {total} strains", end="\r")

        # Stop early if we've exhausted all available pages
        if resp.get("next") is None:
            break

        page += 1
        time.sleep(0.5)   # polite delay between paginated requests

    print(f"\nFetched {len(all_strains)} strain records across {page} page(s).")
    cache_path.write_text(json.dumps(all_strains))
    return all_strains


strains_raw = fetch_strain_pages()
print(f"Total records in memory: {len(strains_raw)}")

### 1.3 Flatten Nested JSON into a Polars DataFrame

In [ ]:
def _first(obj):
    """Return obj if scalar, or first element if list, else None."""
    if isinstance(obj, list):
        return obj[0] if obj else None
    return obj


def flatten_strain(rec: dict) -> dict:
    """
    Flatten one raw BacDive strain record into a flat dict of scalar values.

    BacDive records are deeply nested: phenotypic data live under section keys
    (e.g. ``"Physiology and metabolism"``) whose values are dicts or lists of
    dicts.  We extract the eight most analytically useful fields.

    Parameters
    ----------
    rec : dict
        Raw strain record as returned by the BacDive API.

    Returns
    -------
    dict
        Flat dict with keys: bacdive_id, species_name, gram_stain,
        oxygen_tolerance, isolation_source, temperature_optimum,
        ph_optimum, country.
    """
    # --- Identifiers --------------------------------------------------------
    general = rec.get("General", {})
    bacdive_id = general.get("BacDive-ID")

    # --- Taxonomy -----------------------------------------------------------
    name_block = rec.get("Name and taxonomic classification", {})
    sp = name_block.get("species", {})
    # Handle both dict (single species) and list (multiple synonyms) forms
    if isinstance(sp, list):
        sp = sp[0] if sp else {}
    genus   = sp.get("genus", "")
    epithet = sp.get("species", "")
    species_name = f"{genus} {epithet}".strip() or None

    # --- Morphology: Gram stain ---------------------------------------------
    morph = rec.get("Morphology", {})
    cell_morph = _first(morph.get("cell morphology", {}))
    gram_stain = None
    if isinstance(cell_morph, dict):
        gram_stain = cell_morph.get("gram stain")

    # --- Physiology: oxygen tolerance ---------------------------------------
    physio = rec.get("Physiology and metabolism", {})
    oxy_block = _first(physio.get("oxygen tolerance", {}))
    oxygen_tolerance = None
    if isinstance(oxy_block, dict):
        oxygen_tolerance = oxy_block.get("oxygen tolerance")

    # --- Physiology: temperature optimum ------------------------------------
    temp_block = _first(physio.get("temperature", {}))
    temperature_optimum = None
    if isinstance(temp_block, dict):
        temperature_optimum = temp_block.get("temperature optimum")

    # --- Physiology: pH optimum ---------------------------------------------
    ph_block = _first(physio.get("pH", {}))
    ph_optimum = None
    if isinstance(ph_block, dict):
        ph_optimum = ph_block.get("pH optimum")

    # --- Isolation source and country ---------------------------------------
    isolation = rec.get("Isolation, sampling and environmental information", {})
    iso_block = _first(isolation.get("isolation", {}))
    isolation_source = None
    country = None
    if isinstance(iso_block, dict):
        isolation_source = iso_block.get("isolation source category") or iso_block.get("isolation source")
        country = iso_block.get("country")

    return {
        "bacdive_id":          bacdive_id,
        "species_name":        species_name,
        "gram_stain":          gram_stain,
        "oxygen_tolerance":    oxygen_tolerance,
        "isolation_source":    isolation_source,
        "temperature_optimum": temperature_optimum,
        "ph_optimum":          ph_optimum,
        "country":             country,
    }


# Flatten all cached records into a list of flat dicts
rows = [flatten_strain(r) for r in strains_raw]

# Build Polars DataFrame and cast numeric physiology columns
strains = pl.DataFrame(rows).with_columns([
    pl.col("bacdive_id").cast(pl.Int64, strict=False),
    pl.col("temperature_optimum").cast(pl.Float64, strict=False),
    pl.col("ph_optimum").cast(pl.Float64, strict=False),
])

print(f"Shape  : {strains.shape}")
print(f"\nDtypes:")
print(strains.schema)
print()
strains.head(10)

### 1.4 Fetch Detailed Records for a Genus of Interest (*Escherichia*)

In [ ]:
GENUS_CACHE = DATA_DIR / "bacdive_escherichia.json"
GENUS = "Escherichia"


def fetch_taxon_strains(
    genus: str,
    species: str = "*",
    cache_path: Path = GENUS_CACHE,
) -> list[dict]:
    """
    Fetch all BacDive strain records for a given taxon via the
    ``/taxon/{genus}/{species}/`` endpoint.  Results are paginated and
    cached to disk on first call.

    Parameters
    ----------
    genus : str
        Genus name (e.g. ``"Escherichia"``).
    species : str
        Species epithet, or ``"*"`` to retrieve all species in the genus.
    cache_path : Path
        Destination path for the JSON cache file.

    Returns
    -------
    list[dict]
        Raw BacDive strain record dicts for the requested taxon.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    records: list[dict] = []
    # Taxon endpoint: first call returns count + first page URL
    resp = bacdive_get(f"taxon/{genus}/{species}/")
    records.extend(resp.get("results", []))
    total = resp.get("count", 0)
    print(f"  Found {total} strains for {genus} {species}")

    # Follow 'next' links until exhausted
    next_url = resp.get("next")
    while next_url:
        page_resp = requests.get(next_url, auth=BACDIVE_AUTH, timeout=30)
        page_resp.raise_for_status()
        page_data = page_resp.json()
        records.extend(page_data.get("results", []))
        print(f"  Fetched {len(records)} / {total}", end="\r")
        next_url = page_data.get("next")
        time.sleep(0.5)   # polite delay

    print(f"\nDone. {len(records)} {genus} records retrieved.")
    cache_path.write_text(json.dumps(records))
    return records


escherichia_raw = fetch_taxon_strains(GENUS)
print(f"Raw records: {len(escherichia_raw)}")

### 1.5 DataFrame Shape, Dtypes, and Head

In [ ]:
# Flatten the Escherichia records using the same helper as Section 1.3
ecoli_rows = [flatten_strain(r) for r in escherichia_raw]

escherichia = pl.DataFrame(ecoli_rows).with_columns([
    pl.col("bacdive_id").cast(pl.Int64, strict=False),
    pl.col("temperature_optimum").cast(pl.Float64, strict=False),
    pl.col("ph_optimum").cast(pl.Float64, strict=False),
])

# ── Paginated batch DataFrame ────────────────────────────────────────────────
print("=== strains (paginated batch, all genera) ===")
print(f"Shape  : {strains.shape}")
print(f"Dtypes : {strains.schema}")
print()
print(strains.head(5))

print()

# ── Escherichia genus DataFrame ──────────────────────────────────────────────
print(f"=== escherichia (genus={GENUS}) ===")
print(f"Shape  : {escherichia.shape}")
print(f"Dtypes : {escherichia.schema}")
print()
print(escherichia.head(10))